# Stress Prediction with Explainable ML + Counterfactual + GenAI

Notebook end-to-end untuk penelitian UTS S2 Kecerdasan Komputasional.

**Pipeline**: Predict (CatBoost/RF/TabNet) → Explain (SHAP) → Prescribe (DiCE Counterfactual) → Naturalize (GPT-4o-mini).

**Dataset**: `sleep_health_dataset.csv` (100k baris × 32 kolom, sintetis).

**Cara pakai**:
- Toggle `USE_SAMPLE = True` di Section 0 untuk iterasi cepat dengan 10k sampel.
- Pause points: setelah Section 4 (CatBoost milestone), Section 9 (CF), Section 10 (GenAI first run).
- Section 13: re-run full 100k.

---

## Section 0 — Setup & Konfigurasi


In [ ]:
import os
import json
import time
import joblib
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['savefig.dpi'] = 100

# === KONFIGURASI ===
RANDOM_STATE = 42
USE_SAMPLE = True            # set False di Section 13 untuk full 100k
SAMPLE_SIZE = 10_000

DATA_PATH = Path('sleep_health_dataset.csv')
PROCESSED_DIR = Path('data/processed')
MODELS_DIR = Path('models')
FIGURES_DIR = Path('outputs/figures')
REPORTS_DIR = Path('outputs/reports')
RECOMMENDATIONS_DIR = Path('outputs/recommendations')

for d in [PROCESSED_DIR, MODELS_DIR, FIGURES_DIR, REPORTS_DIR, RECOMMENDATIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# === API KEY (untuk Section 10 saja) ===
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

print(f"USE_SAMPLE       : {USE_SAMPLE}")
print(f"Sample size      : {SAMPLE_SIZE if USE_SAMPLE else 'full 100k'}")
print(f"RANDOM_STATE     : {RANDOM_STATE}")
print(f"OpenAI key loaded: {'OK' if OPENAI_API_KEY else 'MISSING (Section 10 akan di-skip)'}")


## Section 1 — Load & Sampling

Stratified sampling 10k berdasarkan desil `stress_score` untuk preserve distribusi target.


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Full dataset shape: {df_raw.shape}")
print(f"Columns          : {len(df_raw.columns)}")

if USE_SAMPLE:
    bins = pd.qcut(df_raw['stress_score'], q=10, labels=False, duplicates='drop')
    df = (
        df_raw.groupby(bins, group_keys=False)
        .apply(lambda g: g.sample(n=min(len(g), SAMPLE_SIZE // 10), random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )
else:
    df = df_raw.copy()

print(f"Working dataset  : {df.shape}")
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
if missing.sum() == 0:
    print("✓ Tidak ada missing value")
else:
    print(f"Missing total: {missing.sum()}")
    print(missing[missing > 0])


## Section 2 — EDA

Verifikasi statistik draft: mean `stress_score` ≈ 5.73, korelasi `sleep_quality_score` dengan target ≈ -0.639.


In [ ]:
# 2.1 Distribusi target
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['stress_score'], bins=40, edgecolor='black', alpha=0.75, color='steelblue')
ax.axvline(df['stress_score'].mean(), color='red', linestyle='--',
           label=f"Mean = {df['stress_score'].mean():.2f}")
ax.set_xlabel('stress_score')
ax.set_ylabel('Frekuensi')
ax.set_title('Distribusi stress_score')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_stress_distribution.png')
plt.show()

print(f"Mean : {df['stress_score'].mean():.3f}  (target draft ≈ 5.73)")
print(f"Std  : {df['stress_score'].std():.3f}  (target draft ≈ 1.62)")


In [ ]:
# 2.2 Correlation heatmap (numerical only)
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=False, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Correlation Heatmap (Numerical Features)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_correlation_heatmap.png')
plt.show()


In [ ]:
# 2.3 Top korelasi dengan stress_score
target_corr = corr['stress_score'].drop('stress_score').sort_values()
print('Top NEGATIVE correlations with stress_score:')
print(target_corr.head(5).to_string())
print()
print('Top POSITIVE correlations with stress_score:')
print(target_corr.tail(5).to_string())
print()
print(f"  sleep_quality_score corr: {corr.loc['sleep_quality_score', 'stress_score']:.3f}  (target ≈ -0.639)")


In [ ]:
# 2.4 Box plot kategorikal vs target
cat_for_eda = ['gender', 'chronotype', 'mental_health_condition', 'shift_work', 'day_type', 'sleep_aid_used']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, col in zip(axes.flat, cat_for_eda):
    sns.boxplot(data=df, x=col, y='stress_score', ax=ax)
    ax.set_title(f'stress_score by {col}')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'eda_categorical_boxplots.png')
plt.show()


## Section 3 — Preprocessing

**Leakage features di-drop**: `person_id`, `cognitive_performance_score`, `sleep_disorder_risk`, `felt_rested`.

Dua versi data dibuat:
- **A (CatBoost-native)** — kategorikal apa adanya, scaling tidak perlu
- **B (encoded + scaled)** — untuk Random Forest & TabNet

Split stratified 70/15/15 berdasarkan desil `stress_score`.


In [ ]:
# 3.1 Drop leakage features
LEAKAGE_FEATURES = ['person_id', 'cognitive_performance_score', 'sleep_disorder_risk', 'felt_rested']
df_clean = df.drop(columns=LEAKAGE_FEATURES)
print(f"After dropping leakage: {df_clean.shape}")
print(f"Dropped: {LEAKAGE_FEATURES}")


In [ ]:
# 3.2 Identify feature types
TARGET = 'stress_score'
CATEGORICAL_COLS = df_clean.select_dtypes(include=['object']).columns.tolist()
NUMERICAL_COLS = [c for c in df_clean.select_dtypes(include=[np.number]).columns if c != TARGET]

print(f"Categorical ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")
print(f"Numerical   ({len(NUMERICAL_COLS)}): {NUMERICAL_COLS}")


In [ ]:
# 3.3 Stratified split 70/15/15
from sklearn.model_selection import train_test_split

strat_bins = pd.qcut(df_clean[TARGET], q=10, labels=False, duplicates='drop')
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

X_temp, X_test, y_temp, y_test, strat_temp, _ = train_test_split(
    X, y, strat_bins,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=strat_bins,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15 / 0.85,
    random_state=RANDOM_STATE,
    stratify=strat_temp,
)

print(f"Train: {X_train.shape}  |  Val: {X_val.shape}  |  Test: {X_test.shape}")


In [ ]:
# 3.4 Bangun Versi A (catboost) & Versi B (encoded+scaled)
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# Versi A — CatBoost native
X_train_A, X_val_A, X_test_A = X_train.copy(), X_val.copy(), X_test.copy()

# Versi B — Ordinal-encode kategorikal, lalu scale semua
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
encoder.fit(X_train[CATEGORICAL_COLS])

def encode_df(d):
    out = d.copy()
    out[CATEGORICAL_COLS] = encoder.transform(d[CATEGORICAL_COLS])
    return out

X_train_enc = encode_df(X_train)
X_val_enc   = encode_df(X_val)
X_test_enc  = encode_df(X_test)

scaler = StandardScaler()
scaler.fit(X_train_enc)

def to_scaled_df(d):
    return pd.DataFrame(scaler.transform(d), columns=d.columns, index=d.index)

X_train_B = to_scaled_df(X_train_enc)
X_val_B   = to_scaled_df(X_val_enc)
X_test_B  = to_scaled_df(X_test_enc)

# Persist artifacts
joblib.dump(encoder, PROCESSED_DIR / 'ordinal_encoder.pkl')
joblib.dump(scaler, PROCESSED_DIR / 'standard_scaler.pkl')
joblib.dump({
    'X_train_A': X_train_A, 'X_val_A': X_val_A, 'X_test_A': X_test_A,
    'X_train_B': X_train_B, 'X_val_B': X_val_B, 'X_test_B': X_test_B,
    'y_train': y_train, 'y_val': y_val, 'y_test': y_test,
    'CATEGORICAL_COLS': CATEGORICAL_COLS,
    'NUMERICAL_COLS': NUMERICAL_COLS,
}, PROCESSED_DIR / 'splits.pkl')

print(f"Versi A (CatBoost): {X_train_A.shape}")
print(f"Versi B (scaled)  : {X_train_B.shape}")
print(f"Artifacts → {PROCESSED_DIR}")


## Section 4 — CatBoost End-to-End ⭐ MILESTONE 1

Train CatBoost di Versi A. Target validasi: **R² ≥ 0.6, MAE ≤ 1.0**.

**PAUSE** setelah cell ini — review metrik & diagnostic plot sebelum lanjut.


In [ ]:
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

cat_features_idx = [X_train_A.columns.get_loc(c) for c in CATEGORICAL_COLS]

train_pool = Pool(X_train_A, y_train, cat_features=cat_features_idx)
val_pool   = Pool(X_val_A,   y_val,   cat_features=cat_features_idx)
test_pool  = Pool(X_test_A,  y_test,  cat_features=cat_features_idx)

catboost = CatBoostRegressor(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function='RMSE',
    eval_metric='RMSE',
    early_stopping_rounds=50,
    random_seed=RANDOM_STATE,
    verbose=100,
)

t0 = time.time()
catboost.fit(train_pool, eval_set=val_pool, use_best_model=True)
print(f"\nTraining time: {time.time() - t0:.1f}s")
print(f"Best iteration: {catboost.best_iteration_}")


In [ ]:
# Evaluasi test set
def eval_model(model_preds, y_true, name):
    r2   = r2_score(y_true, model_preds)
    rmse = float(np.sqrt(mean_squared_error(y_true, model_preds)))
    mae  = mean_absolute_error(y_true, model_preds)
    print(f"[{name:>14}] R² = {r2:.4f}  |  RMSE = {rmse:.4f}  |  MAE = {mae:.4f}")
    return {'model': name, 'R2': r2, 'RMSE': rmse, 'MAE': mae}

preds_cb = catboost.predict(X_test_A)
cb_results = eval_model(preds_cb, y_test, 'CatBoost')
cb_results['predictions'] = preds_cb

catboost.save_model(MODELS_DIR / 'catboost.cbm')
print(f"Model disimpan: {MODELS_DIR / 'catboost.cbm'}")


In [ ]:
# Diagnostic plots
residuals = y_test.values - preds_cb

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_test, preds_cb, alpha=0.3, s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual stress_score')
axes[0].set_ylabel('Predicted stress_score')
axes[0].set_title('CatBoost: Predicted vs Actual')

axes[1].scatter(preds_cb, residuals, alpha=0.3, s=10)
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')
axes[1].set_title('CatBoost: Residual Plot')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'catboost_diagnostics.png')
plt.show()


In [ ]:
# Milestone check
if cb_results['R2'] >= 0.6 and cb_results['MAE'] <= 1.0:
    print(f"✓ MILESTONE 1 PASS — R² = {cb_results['R2']:.4f}, MAE = {cb_results['MAE']:.4f}")
    print("  Lanjut Section 5.")
elif cb_results['R2'] >= 0.5:
    print(f"⚠ MILESTONE 1 SOFT PASS — R² = {cb_results['R2']:.4f} (di bawah 0.6)")
    print("  Pipeline jalan, tapi pertimbangkan tuning. Boleh lanjut.")
else:
    print(f"✗ MILESTONE 1 FAIL — R² = {cb_results['R2']:.4f}")
    print("  Review preprocessing/leakage sebelum lanjut.")


## Section 5 — Random Forest


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

t0 = time.time()
rf.fit(X_train_B, y_train)
print(f"Training time: {time.time() - t0:.1f}s")

preds_rf = rf.predict(X_test_B)
rf_results = eval_model(preds_rf, y_test, 'RandomForest')
rf_results['predictions'] = preds_rf

joblib.dump(rf, MODELS_DIR / 'random_forest.pkl')


## Section 6 — TabNet

Deep learning untuk tabular data. Lebih lambat dari RF/CatBoost.


In [ ]:
from pytorch_tabnet.tab_model import TabNetRegressor
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"TabNet device: {device}")

tabnet = TabNetRegressor(
    seed=RANDOM_STATE,
    verbose=0,
    device_name=device,
)

X_train_tn = X_train_B.values.astype(np.float32)
X_val_tn   = X_val_B.values.astype(np.float32)
X_test_tn  = X_test_B.values.astype(np.float32)
y_train_tn = y_train.values.reshape(-1, 1).astype(np.float32)
y_val_tn   = y_val.values.reshape(-1, 1).astype(np.float32)

t0 = time.time()
tabnet.fit(
    X_train_tn, y_train_tn,
    eval_set=[(X_val_tn, y_val_tn)],
    eval_metric=['rmse'],
    max_epochs=200,
    patience=20,
    batch_size=1024,
    virtual_batch_size=128,
)
print(f"\nTraining time: {time.time() - t0:.1f}s")

preds_tn = tabnet.predict(X_test_tn).flatten()
tn_results = eval_model(preds_tn, y_test, 'TabNet')
tn_results['predictions'] = preds_tn

tabnet.save_model(str(MODELS_DIR / 'tabnet'))


## Section 7 — Perbandingan Model

Pilih `best_model` berdasarkan R² test. Tambah 5-Fold CV untuk validitas stabilitas pada best model.


In [ ]:
results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != 'predictions'}
    for r in [cb_results, rf_results, tn_results]
]).set_index('model')

print("=== Model Comparison ===")
print(results_df.round(4))
results_df.to_csv(REPORTS_DIR / 'model_comparison.csv')


In [ ]:
# Bar chart komparatif
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, asc in zip(axes, ['R2', 'RMSE', 'MAE'], [False, True, True]):
    sorted_r = results_df[metric].sort_values(ascending=asc)
    bars = ax.bar(sorted_r.index, sorted_r.values, color=['steelblue', 'orange', 'green'])
    ax.set_title(metric)
    ax.set_ylabel(metric)
    for bar, val in zip(bars, sorted_r.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison.png')
plt.show()


In [ ]:
# 5-Fold CV pada best model
from sklearn.model_selection import KFold

best_name = results_df['R2'].idxmax()
print(f"BEST MODEL: {best_name} (R² = {results_df.loc[best_name, 'R2']:.4f})")
print(f"\n5-Fold CV pada {best_name}...\n")

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = []

if best_name == 'CatBoost':
    X_cv = pd.concat([X_train_A, X_val_A])
elif best_name == 'RandomForest':
    X_cv = pd.concat([X_train_B, X_val_B])
else:
    X_cv = pd.concat([X_train_B, X_val_B])
y_cv = pd.concat([y_train, y_val])

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_cv), 1):
    if best_name == 'CatBoost':
        m = CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05,
                              random_seed=RANDOM_STATE, verbose=0)
        m.fit(X_cv.iloc[tr_idx], y_cv.iloc[tr_idx], cat_features=cat_features_idx)
        p = m.predict(X_cv.iloc[va_idx])
    elif best_name == 'RandomForest':
        m = RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                  n_jobs=-1, random_state=RANDOM_STATE)
        m.fit(X_cv.iloc[tr_idx], y_cv.iloc[tr_idx])
        p = m.predict(X_cv.iloc[va_idx])
    else:
        m = TabNetRegressor(seed=RANDOM_STATE, verbose=0, device_name=device)
        m.fit(X_cv.iloc[tr_idx].values.astype(np.float32),
              y_cv.iloc[tr_idx].values.reshape(-1, 1).astype(np.float32),
              max_epochs=80, patience=15, batch_size=1024, virtual_batch_size=128)
        p = m.predict(X_cv.iloc[va_idx].values.astype(np.float32)).flatten()
    fold_r2 = r2_score(y_cv.iloc[va_idx], p)
    cv_scores.append(fold_r2)
    print(f"  Fold {fold}: R² = {fold_r2:.4f}")

print(f"\n5-Fold CV R² = {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")


## Section 8 — SHAP Explainability

Selalu gunakan **CatBoost** untuk SHAP analysis (TreeExplainer paling cepat & stabil),
walaupun best model bisa beda. Hasil SHAP tetap valid untuk explainability pipeline.

- Global: bar + beeswarm (top-15 fitur)
- Local: 3 individu (low/mid/high stress) → waterfall plot
- Domain validity: sign check top-10


In [ ]:
import shap

explainer = shap.TreeExplainer(catboost)

n_shap = min(2000, len(X_test_A))
X_shap = X_test_A.sample(n=n_shap, random_state=RANDOM_STATE)
shap_values = explainer.shap_values(Pool(X_shap, cat_features=cat_features_idx))
print(f"SHAP values shape: {shap_values.shape}")
print(f"Expected value (base): {explainer.expected_value:.4f}")


In [ ]:
# 8.1 Global — bar
plt.figure()
shap.summary_plot(shap_values, X_shap, plot_type='bar', show=False, max_display=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_global_bar.png', bbox_inches='tight')
plt.show()


In [ ]:
# 8.2 Global — beeswarm
plt.figure()
shap.summary_plot(shap_values, X_shap, show=False, max_display=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_global_beeswarm.png', bbox_inches='tight')
plt.show()


In [ ]:
# 8.3 Identify top-10 features
mean_abs_shap = (
    pd.Series(np.abs(shap_values).mean(axis=0), index=X_shap.columns)
    .sort_values(ascending=False)
)
top10 = mean_abs_shap.head(10)
print('Top-10 features by mean |SHAP|:')
print(top10.round(4).to_string())
top10.to_csv(REPORTS_DIR / 'shap_top10_features.csv', header=['mean_abs_shap'])


In [ ]:
# 8.4 Pilih 3 individu (low/mid/high) berdasarkan actual stress_score
y_test_pred = catboost.predict(X_test_A)
target_levels = {'low': 3.0, 'mid': 6.0, 'high': 8.5}
selected_indices = {}

for label, level in target_levels.items():
    diffs = np.abs(y_test.values - level)
    pos = int(np.argmin(diffs))
    selected_indices[label] = pos
    print(f"{label.upper():>4}: pos={pos:5d}  actual={y_test.iloc[pos]:.2f}  pred={y_test_pred[pos]:.2f}")


In [ ]:
# 8.5 Waterfall plot per individu
for label, pos in selected_indices.items():
    x_one = X_test_A.iloc[[pos]]
    sv_one = explainer.shap_values(Pool(x_one, cat_features=cat_features_idx))

    explanation = shap.Explanation(
        values=sv_one[0],
        base_values=explainer.expected_value,
        data=x_one.iloc[0].values,
        feature_names=x_one.columns.tolist(),
    )

    plt.figure()
    shap.plots.waterfall(explanation, max_display=10, show=False)
    plt.title(f'SHAP Waterfall — {label.upper()} stress '
              f'(actual={y_test.iloc[pos]:.2f}, pred={y_test_pred[pos]:.2f})')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f'shap_waterfall_{label}.png', bbox_inches='tight')
    plt.show()


In [ ]:
# 8.6 Domain validity — sign check
print('Domain validity (correlation between feature value and its SHAP):')
print('Expected: sleep_quality ↑ → stress ↓ (negative).\n')
for feat in top10.index[:10]:
    if feat in NUMERICAL_COLS:
        vals = X_shap[feat].values
        svs  = shap_values[:, X_shap.columns.get_loc(feat)]
        c = np.corrcoef(vals, svs)[0, 1]
        print(f"  {feat:35s}: corr(value, SHAP) = {c:+.3f}")


## Section 9 — Counterfactual Analysis (DiCE)

### 9.1 Kategorisasi Fitur (Causal Soundness)

| Kategori | Boleh diubah CF? |
|---|---|
| **Behavior** (sleep_duration, screen_time, caffeine, alcohol, exercise, steps, nap, room_temp, sleep_aid) | ✅ |
| **Outcome** (sleep_quality, wake_episodes, latency, REM, deep_sleep, HR_resting) | ❌ (gejala, bukan kausa) |
| **Immutable** (age, gender, occupation, country, bmi, chronotype, season, shift_work, mental_health, day_type) | ❌ |

### 9.2 Safety constraint — `permitted_range`

| Fitur | Range | Alasan |
|---|---|---|
| sleep_duration_hrs | [4.0, 10.0] | < 4 berbahaya, > 10 indikator depresi |
| caffeine_mg_before_bed | [0, 400] | 400mg = batas aman FDA |
| alcohol_units_before_bed | [0, 0] | Tidak boleh rekomendasi alkohol |
| screen_time_before_bed_mins | [0, 180] | Realistis |
| steps_that_day | [1000, 15000] | Realistis |
| nap_duration_mins | [0, 30] | > 30 ganggu tidur malam |
| room_temperature_celsius | [16, 26] | Range nyaman |


In [ ]:
import dice_ml
from dice_ml import Dice

BEHAVIOR_FEATURES = [
    'sleep_duration_hrs', 'screen_time_before_bed_mins', 'caffeine_mg_before_bed',
    'alcohol_units_before_bed', 'exercise_day', 'steps_that_day',
    'nap_duration_mins', 'room_temperature_celsius', 'sleep_aid_used',
]
OUTCOME_FEATURES = [
    'sleep_quality_score', 'wake_episodes_per_night', 'sleep_latency_mins',
    'rem_percentage', 'deep_sleep_percentage', 'heart_rate_resting_bpm',
]
IMMUTABLE_FEATURES = [
    'age', 'gender', 'occupation', 'country', 'bmi', 'chronotype',
    'season', 'shift_work', 'mental_health_condition', 'day_type',
]

PERMITTED_RANGE = {
    'sleep_duration_hrs':          [4.0, 10.0],
    'caffeine_mg_before_bed':      [0,   400],
    'alcohol_units_before_bed':    [0,   0],
    'screen_time_before_bed_mins': [0,   180],
    'exercise_day':                [0,   1],
    'steps_that_day':              [1000, 15000],
    'nap_duration_mins':           [0,   30],
    'room_temperature_celsius':    [16,  26],
    'sleep_aid_used':              [0,   1],
}

print(f"Behavior  (varyable): {len(BEHAVIOR_FEATURES)}")
print(f"Outcome   (locked)  : {len(OUTCOME_FEATURES)}")
print(f"Immutable (locked)  : {len(IMMUTABLE_FEATURES)}")


In [ ]:
# 9.2 DiCE setup
dice_train_df = X_train_A.copy()
dice_train_df['stress_score'] = y_train.values

continuous_features = [c for c in NUMERICAL_COLS if c != 'stress_score']

dice_data = dice_ml.Data(
    dataframe=dice_train_df,
    continuous_features=continuous_features,
    outcome_name='stress_score',
)
dice_model = dice_ml.Model(model=catboost, backend='sklearn', model_type='regressor')
exp = Dice(dice_data, dice_model, method='random')

print('DiCE Explainer ready (random sampling method).')


In [ ]:
# 9.3 CF Evaluation pada 60 instance (15 per stress quartile)
N_EVAL_PER_QUARTILE = 15

test_with_pred = X_test_A.copy()
test_with_pred['stress_score'] = y_test.values
test_with_pred['_quartile'] = pd.qcut(test_with_pred['stress_score'], q=4, labels=False)

eval_sample = (
    test_with_pred.groupby('_quartile', group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), N_EVAL_PER_QUARTILE), random_state=RANDOM_STATE))
)
eval_X = eval_sample.drop(columns=['stress_score', '_quartile'])
print(f"Evaluating CF on {len(eval_X)} instances...")


In [ ]:
def normalize_l1(orig, cf_row, ranges):
    diff, n = 0.0, 0
    for f, (lo, hi) in ranges.items():
        if f in orig.index and f in cf_row.index:
            span = max(hi - lo, 1e-6)
            diff += abs(float(orig[f]) - float(cf_row[f])) / span
            n += 1
    return diff / max(n, 1)

cf_metrics_rows = []

for i, (idx, query) in enumerate(eval_X.iterrows()):
    orig_pred = float(catboost.predict(query.to_frame().T)[0])
    target_hi = max(1.0, orig_pred - 1.5)

    try:
        cf = exp.generate_counterfactuals(
            query.to_frame().T,
            total_CFs=3,
            desired_range=[1.0, target_hi],
            features_to_vary=BEHAVIOR_FEATURES,
            permitted_range=PERMITTED_RANGE,
            verbose=False,
        )
        cf_df = cf.cf_examples_list[0].final_cfs_df
        if cf_df is None or len(cf_df) == 0:
            continue

        cf_preds = cf_df['stress_score'].values
        valid_count = int(np.sum(cf_preds <= target_hi))

        sparsities, proximities, plausibilities = [], [], []
        for j in range(len(cf_df)):
            cf_row = cf_df.iloc[j]
            changed = [
                f for f in BEHAVIOR_FEATURES
                if not np.isclose(float(cf_row[f]), float(query[f]), rtol=0.01)
            ]
            sparsities.append(len(changed))
            proximities.append(normalize_l1(query, cf_row, PERMITTED_RANGE))
            in_range = all(
                PERMITTED_RANGE[f][0] <= float(cf_row[f]) <= PERMITTED_RANGE[f][1]
                for f in BEHAVIOR_FEATURES
            )
            plausibilities.append(int(in_range))

        diversities = []
        if len(cf_df) > 1:
            for a in range(len(cf_df)):
                for b in range(a + 1, len(cf_df)):
                    diversities.append(normalize_l1(cf_df.iloc[a], cf_df.iloc[b], PERMITTED_RANGE))

        cf_metrics_rows.append({
            'instance_idx': idx,
            'orig_pred':    orig_pred,
            'n_cfs':        len(cf_df),
            'validity':     valid_count / len(cf_df),
            'proximity':    float(np.mean(proximities)),
            'sparsity':     float(np.mean(sparsities)),
            'diversity':    float(np.mean(diversities)) if diversities else 0.0,
            'plausibility': float(np.mean(plausibilities)),
            'quartile':     int(eval_sample.loc[idx, '_quartile']),
        })
    except Exception as e:
        print(f"  [{i}] idx={idx} failed: {type(e).__name__}: {e}")

    if (i + 1) % 15 == 0:
        print(f"  Processed {i + 1}/{len(eval_X)}...")

cf_metrics_df = pd.DataFrame(cf_metrics_rows)
print(f"\nCF metrics for {len(cf_metrics_df)} instances:")
print(cf_metrics_df[['validity', 'proximity', 'sparsity', 'diversity', 'plausibility']].mean().round(3))
cf_metrics_df.to_csv(REPORTS_DIR / 'cf_evaluation_metrics.csv', index=False)


In [ ]:
# 9.3.1 Boxplots per quartile
if len(cf_metrics_df) > 0:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, metric in zip(axes.flat, ['validity', 'proximity', 'sparsity', 'diversity', 'plausibility']):
        sns.boxplot(data=cf_metrics_df, x='quartile', y=metric, ax=ax)
        ax.set_title(f'CF {metric} by stress quartile')
    axes.flat[-1].axis('off')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'cf_metrics_by_quartile.png')
    plt.show()
else:
    print('No CF metrics to plot.')


In [ ]:
# 9.4 Case study — 3 individu (low/mid/high)
case_study_results = {}

for label, pos in selected_indices.items():
    query = X_test_A.iloc[[pos]]
    actual_y  = float(y_test.iloc[pos])
    orig_pred = float(catboost.predict(query)[0])
    target_hi = max(1.0, orig_pred - 1.5)

    print(f"\n=== {label.upper()} STRESS individu ===")
    print(f"Actual: {actual_y:.2f}  |  Predicted: {orig_pred:.2f}  |  Target CF: < {target_hi:.2f}")

    try:
        cf = exp.generate_counterfactuals(
            query,
            total_CFs=3,
            desired_range=[1.0, target_hi],
            features_to_vary=BEHAVIOR_FEATURES,
            permitted_range=PERMITTED_RANGE,
            verbose=False,
        )
        cf_df = cf.cf_examples_list[0].final_cfs_df
    except Exception as e:
        print(f"  CF generation failed: {e}")
        continue

    # Pilih CF dengan perubahan paling minimal (sparsity terendah)
    best_cf_row, best_changes, best_sparsity = None, [], 10**9
    for j in range(len(cf_df)):
        cf_row = cf_df.iloc[j]
        changes = []
        for f in BEHAVIOR_FEATURES:
            if not np.isclose(float(cf_row[f]), float(query.iloc[0][f]), rtol=0.01):
                changes.append({
                    'feature': f,
                    'before':  float(query.iloc[0][f]),
                    'after':   float(cf_row[f]),
                })
        if len(changes) < best_sparsity:
            best_sparsity = len(changes)
            best_cf_row   = cf_row
            best_changes  = changes

    print(f"Best CF predicted stress = {float(best_cf_row['stress_score']):.2f}")
    print(f"Changes ({len(best_changes)}):")
    for ch in best_changes:
        print(f"  • {ch['feature']:30s}: {ch['before']:.2f} → {ch['after']:.2f}")

    # convert query to dict with serializable types
    query_dict = {}
    for k, v in query.iloc[0].to_dict().items():
        query_dict[k] = float(v) if isinstance(v, (np.floating, np.integer)) else v
    case_study_results[label] = {
        'pos_in_test':  pos,
        'actual':       actual_y,
        'orig_pred':    orig_pred,
        'cf_pred':      float(best_cf_row['stress_score']),
        'delta':        float(best_cf_row['stress_score']) - orig_pred,
        'changes':      best_changes,
        'query':        query_dict,
    }

with open(REPORTS_DIR / 'cf_case_studies.json', 'w', encoding='utf-8') as f:
    json.dump(case_study_results, f, indent=2, ensure_ascii=False, default=str)
print(f"\nCase studies saved → {REPORTS_DIR / 'cf_case_studies.json'}")


## Section 10 — GenAI Naturalisasi 🆕

Ubah angka counterfactual + SHAP menjadi narasi sistem pakar (JSON terstruktur):
`summary` + `drivers` + `recommendations` (3 langkah) + `encouragement` + `disclaimer`.

**Safety**: post-generation regex filter untuk kata terlarang (obat, diagnosa, janji pasti, dll.) → retry maks 3x.

Jika `OPENAI_API_KEY` belum ada di `.env`, section ini di-skip dengan dummy output.


In [ ]:
from openai import OpenAI

with open('prompts/expert_system_prompt.md', 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read()
print(f"System prompt loaded: {len(SYSTEM_PROMPT)} chars")

if OPENAI_API_KEY:
    client = OpenAI(api_key=OPENAI_API_KEY)
    print('OpenAI client ready.')
else:
    client = None
    print('⚠ OPENAI_API_KEY tidak ditemukan. Section 10 akan generate dummy output.')


In [ ]:
def top5_shap_for(pos_in_test):
    x_one = X_test_A.iloc[[pos_in_test]]
    sv_one = explainer.shap_values(Pool(x_one, cat_features=cat_features_idx))[0]
    abs_series = pd.Series(np.abs(sv_one), index=x_one.columns).sort_values(ascending=False)
    top5_feats = abs_series.head(5).index.tolist()
    out = []
    for f in top5_feats:
        v = x_one.iloc[0][f]
        sv = float(sv_one[x_one.columns.get_loc(f)])
        out.append({
            'feature':   f,
            'value':     float(v) if isinstance(v, (np.floating, np.integer)) else v,
            'shap':      round(sv, 4),
            'direction': 'meningkatkan stres' if sv > 0 else 'menurunkan stres',
        })
    return out


def build_user_prompt(case):
    profile_keys = ['age', 'gender', 'occupation']
    profile = {k: case['query'].get(k) for k in profile_keys}
    top5 = top5_shap_for(case['pos_in_test'])
    payload = {
        'profil':                  profile,
        'prediksi_stress_score':   round(case['orig_pred'], 2),
        'cf_target_stress_score':  round(case['cf_pred'], 2),
        'top5_faktor_pengaruh':    top5,
        'perubahan_disarankan':    case['changes'],
    }
    return (
        'Berikut data analisis machine learning untuk satu individu. '
        'Buat narasi rekomendasi sesuai aturan sistem prompt. '
        'JANGAN menambah fakta di luar data ini.\n\n'
        f"```json\n{json.dumps(payload, indent=2, ensure_ascii=False, default=str)}\n```"
    )


FORBIDDEN_WORDS = [
    'obat', 'melatonin', 'antidepresan', 'valerian', ' cbd',
    'suplemen', 'diagnosa', 'didiagnosis', 'menyembuhkan',
    'menjamin', 'akan menurunkan', 'pasti turun', 'pasti hilang',
    'minum bir', 'minum wine', 'konsumsi alkohol',
]
def safety_check(text):
    t = text.lower()
    return [w for w in FORBIDDEN_WORDS if w in t]


def dummy_output(label):
    return {
        'summary': f"[DUMMY {label}] Output placeholder — isi OPENAI_API_KEY di .env lalu re-run.",
        'drivers': ['placeholder driver 1', 'placeholder driver 2', 'placeholder driver 3'],
        'recommendations': [
            {'action': 'placeholder action', 'target': 'placeholder target', 'rationale': 'placeholder rationale'}
            for _ in range(3)
        ],
        'encouragement': 'placeholder encouragement',
        'disclaimer': 'Rekomendasi ini bersifat informatif berdasarkan pola data penelitian. Untuk kondisi medis, konsultasikan dengan profesional kesehatan.',
    }


In [ ]:
genai_outputs = {}

for label, case in case_study_results.items():
    print(f"\n=== Generating untuk {label.upper()} ===")

    if client is None:
        genai_outputs[label] = dummy_output(label)
        print('  (dummy — no API key)')
    else:
        user_prompt = build_user_prompt(case)

        for attempt in range(1, 4):
            try:
                resp = client.chat.completions.create(
                    model='gpt-4o-mini',
                    messages=[
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': user_prompt},
                    ],
                    temperature=0.3,
                    max_tokens=700,
                    response_format={'type': 'json_object'},
                )
                content = resp.choices[0].message.content
                parsed = json.loads(content)
            except json.JSONDecodeError:
                print(f"  Attempt {attempt}: invalid JSON, retry...")
                continue
            except Exception as e:
                print(f"  Attempt {attempt}: API error {type(e).__name__}: {e}")
                break

            flags = safety_check(content)
            if flags:
                print(f"  Attempt {attempt}: safety flags {flags}, retry...")
                continue

            required = {'summary', 'drivers', 'recommendations', 'encouragement', 'disclaimer'}
            if not required.issubset(parsed.keys()):
                print(f"  Attempt {attempt}: missing keys {required - parsed.keys()}, retry...")
                continue

            genai_outputs[label] = parsed
            print(f"  ✓ Generated OK on attempt {attempt}")
            break
        else:
            print(f"  ✗ Failed after 3 attempts — using dummy")
            genai_outputs[label] = dummy_output(label)

    out_path = RECOMMENDATIONS_DIR / f'individual_{label}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(genai_outputs[label], f, indent=2, ensure_ascii=False)
    print(f"  → saved {out_path}")


In [ ]:
# Render markdown untuk inline review
from IPython.display import Markdown, display

for label, output in genai_outputs.items():
    case = case_study_results[label]
    md_text = f"### {label.upper()} STRESS — Rekomendasi Naratif\n\n"
    md_text += f"_actual = {case['actual']:.2f}, predicted = {case['orig_pred']:.2f}, "
    md_text += f"CF target = {case['cf_pred']:.2f} (Δ = {case['delta']:+.2f})_\n\n"
    md_text += f"**Ringkasan**: {output['summary']}\n\n"
    md_text += "**Faktor Utama**:\n"
    for d in output['drivers']:
        md_text += f"- {d}\n"
    md_text += "\n**Langkah Konkret**:\n"
    for r in output['recommendations']:
        md_text += f"- **{r['action']}** — {r['target']}\n"
        md_text += f"  - *Alasan*: {r['rationale']}\n"
    md_text += f"\n**Semangat**: {output['encouragement']}\n\n"
    md_text += f"> _{output['disclaimer']}_\n\n---\n"
    display(Markdown(md_text))


## Section 11 — Individual Insights & Kesimpulan


In [ ]:
insights_md = "# Individual Insights — Stress Prediction\n\n"
insights_md += f"**Model terbaik**: `{best_name}` (R² = {results_df.loc[best_name, 'R2']:.4f}, "
insights_md += f"RMSE = {results_df.loc[best_name, 'RMSE']:.4f}, MAE = {results_df.loc[best_name, 'MAE']:.4f})\n\n"
insights_md += "## Top-5 Fitur Paling Berpengaruh (SHAP global)\n\n"
for i, (f, v) in enumerate(top10.head(5).items(), 1):
    insights_md += f"{i}. `{f}` — mean |SHAP| = {v:.3f}\n"
insights_md += "\n---\n\n"

for label in ['low', 'mid', 'high']:
    if label not in case_study_results:
        continue
    case = case_study_results[label]
    insights_md += f"## Individu {label.upper()} Stress\n\n"
    insights_md += f"- **Actual stress_score**: {case['actual']:.2f}\n"
    insights_md += f"- **Predicted**: {case['orig_pred']:.2f}\n"
    insights_md += f"- **CF target prediction**: {case['cf_pred']:.2f} (Δ = {case['delta']:+.2f})\n\n"
    insights_md += f"**Perubahan disarankan ({len(case['changes'])} fitur):**\n"
    for ch in case['changes']:
        insights_md += f"- `{ch['feature']}`: {ch['before']:.2f} → {ch['after']:.2f}\n"
    if label in genai_outputs:
        insights_md += f"\n**Narasi GenAI**:\n\n> {genai_outputs[label]['summary']}\n\n"
    insights_md += "---\n\n"

insights_md += "## Kontribusi Penelitian\n\n"
insights_md += "1. **Explainable** — SHAP mengidentifikasi driver utama stres per individu, mendukung interpretabilitas model.\n"
insights_md += "2. **Prescriptive** — DiCE Counterfactual memberi rekomendasi perubahan **behavior** minimal & realistis (causal-sound, di-constrain dengan `permitted_range`).\n"
insights_md += "3. **Naturalized** — GenAI (GPT-4o-mini) mengubah angka teknis menjadi narasi sistem pakar empatik berbahasa Indonesia, dengan safety filter.\n"

with open(REPORTS_DIR / 'individual_insights.md', 'w', encoding='utf-8') as f:
    f.write(insights_md)

from IPython.display import Markdown, display
print(f"Saved → {REPORTS_DIR / 'individual_insights.md'}\n")
display(Markdown(insights_md))


## Section 12 — Limitations & Threats to Validity 🆕

Section eksplisit yang membahas keterbatasan jujur — wajib untuk paper akademik.


In [ ]:
limitations_md = '''# Limitations & Threats to Validity

## 12.1 Dataset Sintetis

Dataset Sleep Health & Daily Performance (Kaggle) adalah **data sintetis**, bukan pengukuran riil dari subjek manusia. Korelasi dan pola yang ditemukan mungkin merupakan artefak generator data, bukan fenomena alami.

**Disclaimer**: Penelitian ini adalah **methodological proof-of-concept**, bukan validasi klinis. Generalisasi ke populasi nyata memerlukan validasi pada data klinis.

## 12.2 Asumsi Kausalitas pada Counterfactual

DiCE mengasumsikan **causal stationarity** — perubahan fitur akan menghasilkan perubahan prediksi secara konsisten. Pada data sintetis, asumsi ini belum tentu valid. Rekomendasi intervensi harus dipahami sebagai "kemungkinan", bukan "kepastian". Validasi kausal nyata memerlukan eksperimen prospektif (RCT).

Untuk mitigasi parsial, kami:
- Memisahkan **behavior** (boleh diubah) vs **outcome** (gejala, tidak boleh) vs **immutable**
- Menerapkan `permitted_range` eksplisit per fitur behavior

## 12.3 GenAI Evaluation Scope

Evaluasi GenAI terbatas pada:
- **Struktur output**: JSON valid + key wajib (`summary`, `drivers`, `recommendations`, `encouragement`, `disclaimer`)
- **Safety filter**: regex blocklist untuk kata terlarang (obat, diagnosa, janji pasti, alkohol)

**Tidak dilakukan**: user study, expert panel, atau validasi klinis output GenAI. GenAI diposisikan sebagai **deployment layer** untuk presentasi naratif, bukan kontribusi ilmiah utama. Evaluasi rigor LLM (faithfulness scoring lewat human raters, hallucination benchmark) di luar scope penelitian ini.

## 12.4 Reproducibility Bound

`random_state=42` digunakan konsisten di seluruh pipeline (sampling, split, model). Namun:
- **DiCE** menggunakan `method='random'` — meskipun seed di-set, hasil bisa sedikit bervariasi karena sampling internal
- **GPT** bersifat probabilistik meski `temperature=0.3` — variasi minor antar run masih mungkin terjadi
- **TabNet** sensitif terhadap inisialisasi weight — gunakan multiple seeds untuk publikasi

## 12.5 Sampling Bias

Hasil pada sampel 10k bisa berbeda dari full 100k. Section 13 menjalankan ulang full untuk validasi final.
'''
with open(REPORTS_DIR / 'limitations.md', 'w', encoding='utf-8') as f:
    f.write(limitations_md)

from IPython.display import Markdown, display
print(f"Saved → {REPORTS_DIR / 'limitations.md'}\n")
display(Markdown(limitations_md))


## Section 13 — Re-run Full 100k

Setelah validasi pipeline pada sample 10k berhasil:

1. **Restart kernel** (`Kernel → Restart`)
2. Edit Section 0 — ubah `USE_SAMPLE = False`
3. **Run All Cells** (`Cell → Run All`)
4. Tunggu — TabNet & DiCE jauh lebih lambat pada 100k (estimasi: 30–60 menit total tergantung CPU/GPU)

**Catatan**:
- Output (`outputs/figures/`, `models/`, `outputs/reports/`) akan **ter-overwrite**.
- Backup folder dulu jika ingin membandingkan sample vs full.
- GPT call cost tetap rendah (~$0.05) karena hanya 3 individu × maks 3 retry.
